# Primal-Dual SVM: constrained optimization in two views

This academic notebook derives the soft-margin SVM primal and Lagrange dual, implements a primal subgradient method and dual quadratic-program solve, and validates both against `sklearn.svm.SVC`. The data are the real Wisconsin Diagnostic Breast Cancer observations bundled by scikit-learn.

## 1. Primal problem and KKT conditions

For labels $y_i\in\{-1,+1\}$ and features $x_i$, the soft-margin primal is

$$\min_{w,b,\xi}\;\frac12\|w\|_2^2+C\sum_i\xi_i\quad\text{s.t.}\quad y_i(w^Tx_i+b)\ge1-\xi_i,\;\xi_i\ge0.$$

Eliminating $\xi_i$ gives the hinge objective $\frac12\|w\|^2+C\sum_i\max(0,1-y_i(w^Tx_i+b))$. The constraints are convex and Slater's condition holds, so the KKT conditions are necessary and sufficient. The stationarity equations yield the dual

$$\max_{\alpha}\;\mathbf1^T\alpha-\frac12\alpha^TQ\alpha,\quad Q_{ij}=y_iy_jx_i^Tx_j,$$

with $0\le\alpha_i\le C$ and $y^T\alpha=0$. Complementary slackness identifies support vectors and recovers $w=\sum_i\alpha_i y_i x_i$.

## 2. Course methods and convergence

The primal hinge loss is non-smooth, so ordinary Newton assumptions do not apply at margin points. We use a diminishing-step subgradient method (a convex, non-smooth extension of gradient descent), recording objective values and classification accuracy. The smooth dual is solved by SLSQP with analytic objective and gradient, while `SVC(kernel='linear')` is an independent reference implementation. For a convex primal/dual pair, any feasible primal-dual gap is a global certificate; optimizer success codes and KKT residuals are inspected explicitly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

data = load_breast_cancer(as_frame=True)
X = StandardScaler().fit_transform(data.data)
y = np.where(data.target.to_numpy() == 0, -1.0, 1.0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=2026, stratify=y)
C = 0.01
n = X_train.shape[0]

In [ ]:
def primal_subgradient(X, y, C=0.01, iterations=2500):
    w = np.zeros(X.shape[1]); b = 0.0; history = []
    for k in range(1, iterations + 1):
        margins = y * (X @ w + b)
        violating = margins < 1.0
        objective = 0.5 * (w @ w) + C * np.maximum(0.0, 1.0 - margins).sum()
        history.append(objective)
        step = 0.2 / np.sqrt(k)
        w -= step * (w - C * (X[violating].T @ y[violating]))
        b -= step * (-C * y[violating].sum())
    return w, b, np.asarray(history)

w_primal, b_primal, primal_history = primal_subgradient(X_train, y_train, C=C)
plt.plot(primal_history); plt.yscale('log'); plt.xlabel('Iteration'); plt.ylabel('Primal objective'); plt.title('Diminishing-step primal subgradient'); plt.tight_layout();

In [ ]:
Q = (y_train[:, None] * X_train) @ (y_train[:, None] * X_train).T
def dual_objective(alpha):
    return -(alpha.sum() - 0.5 * alpha @ Q @ alpha)
def dual_gradient(alpha):
    return -(np.ones(n) - Q @ alpha)
dual = minimize(dual_objective, np.zeros(n), jac=dual_gradient, method='SLSQP',
                bounds=[(0.0, C)] * n, constraints={'type': 'eq', 'fun': lambda a: y_train @ a, 'jac': lambda a: y_train},
                options={'ftol': 1e-10, 'maxiter': 1000})
alpha = dual.x
w_dual = (alpha * y_train) @ X_train
support = alpha > 1e-6
margin_support = (alpha > 1e-6) & (alpha < C - 1e-6)
b_dual = np.mean(y_train[margin_support] - X_train[margin_support] @ w_dual) if margin_support.any() else 0.0
print('Dual success:', dual.success, '| support vectors:', support.sum(), '| equality residual:', y_train @ alpha)

In [ ]:
reference = SVC(C=C, kernel='linear').fit(X_train, (y_train > 0).astype(int))
pred_primal = np.where(X_test @ w_primal + b_primal >= 0, 1.0, -1.0)
pred_dual = np.where(X_test @ w_dual + b_dual >= 0, 1.0, -1.0)
pred_reference = np.where(reference.predict(X_test) > 0, 1.0, -1.0)
comparison = {
    'primal subgradient accuracy': accuracy_score(y_test, pred_primal),
    'dual SLSQP accuracy': accuracy_score(y_test, pred_dual),
    'SVC reference accuracy': accuracy_score(y_test, pred_reference),
    '||w_primal-w_dual||': np.linalg.norm(w_primal - w_dual),
    'dual success': dual.success,
}
comparison

## 3. Interpretation

The dual equality constraint and box constraints are the numerical expression of the KKT geometry. Agreement of the dual classifier with the reference solver is more informative than accuracy alone: it checks the optimization formulation, support-vector structure and recovered separating hyperplane. The course's Newton/BFGS smooth-convergence theorems are not invoked for the hinge kink; the subgradient method is reported with its appropriate non-smooth qualification.